# EDA — ChemAI: Predict the Cure

Исследовательский анализ данных: распределения таргетов, инсайты, признаки.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

TARGET_IC50 = 'IC50, mM'
TARGET_CC50 = 'CC50, mM'
TARGET_SI   = 'SI'
TARGETS     = [TARGET_IC50, TARGET_CC50, TARGET_SI]

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')

## 1. Распределения таргетов

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(TARGETS):
    axes[0, i].hist(train[col], bins=40, edgecolor='k', alpha=0.7)
    axes[0, i].set_title(f'{col} (original)')
    axes[0, i].set_xlabel('mM')

    axes[1, i].hist(np.log(train[col]), bins=40, edgecolor='k', alpha=0.7, color='orange')
    axes[1, i].set_title(f'log({col})')

plt.tight_layout()
plt.show()

print('Статистика таргетов:')
print(train[TARGETS].describe().round(2))

## 2. Инвариант SI = CC50 / IC50

In [ ]:
si_calc = train[TARGET_CC50] / train[TARGET_IC50]
err     = (train[TARGET_SI] - si_calc).abs()
print(f'SI = CC50/IC50: максимальная ошибка = {err.max():.2e}')
print(f'Совпадение (err < 1e-6): {(err < 1e-6).mean():.1%}')
print('Вывод: log(SI) = log(CC50) - log(IC50) — используем как дополнительный признак и для бленда')

## 3. Пропуски и константные признаки

In [ ]:
feat_cols = [c for c in train.columns if c not in ['index'] + TARGETS]

missing = train[feat_cols].isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f'Признаков с пропусками: {len(missing)}')
print(missing)

const = [c for c in feat_cols if train[c].nunique() <= 1]
print(f'\nКонстантных признаков: {len(const)}')

## 4. Корреляция с таргетами (топ признаков)

In [ ]:
feat_valid = [c for c in feat_cols if c not in const]
X = train[feat_valid].fillna(train[feat_valid].median())

for name, target in [('IC50', TARGET_IC50), ('CC50', TARGET_CC50)]:
    corr = X.corrwith(np.log(train[target])).abs().sort_values(ascending=False)
    print(f'Топ-10 по |corr| с log({name}):')
    print(corr.head(10).to_string())
    print()

## 5. Мультиколлинеарность

In [ ]:
corr_m = X.corr().abs()
upper  = corr_m.where(np.triu(np.ones(corr_m.shape), k=1).astype(bool))

for threshold in [0.99, 0.98, 0.95, 0.90]:
    n_drop = sum(any(upper[c] > threshold) for c in upper.columns)
    print(f'r > {threshold}: {n_drop} признаков к удалению')

print('\nВывод: порог r>0.95 даёт хороший баланс между удалением шума и сохранением сигнала')